In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

In [3]:
PROJECT_DIR = Path("/Users/hyuna/openNeuro/ds007116")
OUTPUT_DIR = PROJECT_DIR / "derived"

analysis_df = pd.read_csv(
    OUTPUT_DIR / "phenotype_merged_long.csv"
)

print(analysis_df.shape)
analysis_df.head()

(224, 13)


,participant_id,session_id,eswan_adhd_inattention_total,eswan_adhd_hyperactivity_impulsivity_total,bissc_total,bas_total,im_average,em_average,cnb_cptnl_total_sen,cnb_cptnl_total_fp,cnb_edisc_tot,cnb_ddisc_tot,cnb_rdisc_tot
0,sub-138950,ses-1,13.0,9.0,14.0,25.0,4.142857,5.333333,0.9000,13.0,62.0,10.0,20.0
1,sub-138950,ses-2,16.0,24.0,16.0,29.0,2.857143,4.166667,0.8667,6.0,74.0,14.0,25.0
2,sub-19861,ses-1,-5.0,-4.0,15.0,17.0,4.285714,4.833333,0.9500,2.0,57.0,6.0,36.0
3,sub-19861,ses-2,NaN,NaN,23.0,17.0,NaN,NaN,0.9833,2.0,60.0,3.0,22.0
4,sub-20124,ses-1,-17.0,-10.0,19.0,24.0,NaN,NaN,NaN,NaN,58.0,12.0,32.0


In [4]:
FEATURES = [
    "eswan_adhd_inattention_total",
    "eswan_adhd_hyperactivity_impulsivity_total",
    "bissc_total",
    "bas_total",
    "im_average",
    "em_average",
    "cnb_cptnl_total_sen",
    "cnb_cptnl_total_fp",
    "cnb_edisc_tot",
    "cnb_ddisc_tot",
    "cnb_rdisc_tot",
]

In [10]:
# ---------------------------------------------------------
# Baseline (ses-1) selection
# ---------------------------------------------------------

baseline_df = analysis_df.loc[
    analysis_df["session_id"] == "ses-1",
    ["participant_id", "session_id"] + FEATURES
].copy()

print("Baseline rows before complete-case selection:", len(baseline_df))
print(
    "Baseline participants before complete-case selection:",
    baseline_df["participant_id"].nunique()
)

# Check participant-level uniqueness
duplicates = baseline_df.duplicated(
    subset="participant_id",
    keep=False
)

print("Duplicated participants at baseline:", duplicates.sum())

Baseline rows before complete-case selection: 132
Baseline participants before complete-case selection: 132
Duplicated participants at baseline: 0


In [11]:
# ---------------------------------------------------------
# Complete-case selection across the 11 phenotype features
# ---------------------------------------------------------

complete_mask = baseline_df[FEATURES].notna().all(axis=1)

feature_complete = baseline_df.loc[
    complete_mask
].copy()

print("Complete baseline rows:", len(feature_complete))
print(
    "Unique participants:",
    feature_complete["participant_id"].nunique()
)

Complete baseline rows: 92
Unique participants: 92


In [12]:
assert len(feature_complete) == 92
assert feature_complete["participant_id"].nunique() == 92

In [ ]:
## Baseline sample selection

Penn LEAD contains repeated assessments across up to three sessions.
The present study focused on cross-sectional attention/motivation
phenotypes; therefore, only baseline (`ses-1`) observations were used
for phenotype construction.

Participants were subsequently required to have complete data across
all 11 clustering indicators. This resulted in a final phenotype
sample of N = 92, with one observation per participant.

In [14]:
desc = feature_complete[FEATURES].describe().T

desc[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

,count,mean,std,min,25%,50%,75%,max
eswan_adhd_inattention_total,92.0,1.521739,9.201622,-27.000000,-4.000000,1.500000,8.250000,21.0
eswan_adhd_hyperactivity_impulsivity_total,92.0,1.836957,9.494101,-27.000000,-4.000000,1.000000,7.250000,27.0
bissc_total,92.0,16.641304,3.853303,7.000000,14.000000,16.000000,20.000000,27.0
bas_total,92.0,24.858696,6.927539,13.000000,21.000000,24.000000,28.250000,51.0
im_average,92.0,4.590062,1.528413,1.000000,3.428571,4.642857,5.892857,7.0
em_average,92.0,4.846014,1.210262,1.166667,4.000000,5.000000,5.666667,7.0
cnb_cptnl_total_sen,92.0,0.844023,0.146623,0.166700,0.800000,0.883300,0.933300,1.0
cnb_cptnl_total_fp,92.0,18.097826,15.128795,2.000000,9.750000,13.000000,22.000000,95.0
cnb_edisc_tot,92.0,63.184783,7.477335,50.000000,58.000000,62.000000,67.250000,84.0
cnb_ddisc_tot,92.0,10.913043,5.381393,0.000000,7.000000,11.000000,13.000000,32.0


In [15]:
scaler = StandardScaler()

X = scaler.fit_transform(
    feature_complete[FEATURES]
)

X_z = pd.DataFrame(
    X,
    columns=FEATURES,
    index=feature_complete.index
)

X_z.describe().T

,count,mean,std,min,25%,50%,75%,max
eswan_adhd_inattention_total,92.0,-3.861645e-17,1.005479,-3.116627,-0.603371,-0.002375,0.735210,2.128428
eswan_adhd_hyperactivity_impulsivity_total,92.0,1.093630e-18,1.005479,-3.053998,-0.618167,-0.088638,0.573272,2.664910
bissc_total,92.0,2.606611e-16,1.005479,-2.515798,-0.689221,-0.167342,0.876417,2.702994
bas_total,92.0,-1.580861e-16,1.005479,-1.721199,-0.560060,-0.124633,0.492222,3.794211
im_average,92.0,-2.075634e-16,1.005479,-2.361752,-0.764096,0.034732,0.857055,1.585398
em_average,92.0,1.291238e-16,1.005479,-3.056784,-0.702865,0.127930,0.681794,1.789520
cnb_cptnl_total_sen,92.0,-9.889432e-16,1.005479,-4.644792,-0.301890,0.269346,0.612225,1.069625
cnb_cptnl_total_fp,92.0,-1.224866e-16,1.005479,-1.069883,-0.554807,-0.338808,0.259344,5.111019
cnb_edisc_tot,92.0,4.537433e-16,1.005479,-1.772962,-0.697199,-0.159318,0.546651,2.799029
cnb_ddisc_tot,92.0,6.999232e-17,1.005479,-2.039034,-0.731128,0.016247,0.389935,3.939966


In [16]:
extreme_counts = (np.abs(X_z) > 3).sum()

extreme_counts[extreme_counts > 0].sort_values(
    ascending=False
)

cnb_cptnl_total_sen                           4
cnb_cptnl_total_fp                            2
cnb_ddisc_tot                                 2
eswan_adhd_inattention_total                  1
eswan_adhd_hyperactivity_impulsivity_total    1
bas_total                                     1
em_average                                    1
cnb_rdisc_tot                                 1
dtype: int64

In [17]:
extreme_rows = X_z[
    (np.abs(X_z) > 3).any(axis=1)
]

print("Rows with at least one |z| > 3:", len(extreme_rows))

extreme_rows

Rows with at least one |z| > 3: 11


,eswan_adhd_inattention_total,eswan_adhd_hyperactivity_impulsivity_total,bissc_total,bas_total,im_average,em_average,cnb_cptnl_total_sen,cnb_cptnl_total_fp,cnb_edisc_tot,cnb_ddisc_tot,cnb_rdisc_tot
52,-0.821915,-1.783130,-0.428281,1.036506,-1.985833,-0.702865,-3.616155,-0.737576,1.319855,0.576778,0.660038
54,0.270804,-0.194544,0.876417,2.052502,-0.764096,-1.256728,-0.416411,-0.072963,0.109623,-2.039034,-3.154251
93,-0.057011,0.440890,0.354538,0.601079,0.551620,0.266396,-3.159441,-0.205886,0.916444,1.510997,1.728038
121,0.489348,1.076324,0.876417,-0.560060,0.739580,-0.287467,0.612225,-0.737576,-0.831670,3.753122,0.202323
131,-3.116627,-3.053998,-1.732979,-1.721199,-1.515935,-1.118262,-0.416411,-0.604653,0.378563,0.576778,-0.560535
136,-1.259003,-0.300450,1.659236,3.794211,-2.079813,-3.056784,-1.445047,2.120260,-1.772962,-1.478503,0.202323
139,0.380076,1.817664,0.354538,-0.269775,-0.952056,-0.010535,-4.644792,-1.069883,1.050915,0.389935,-0.865678
153,-1.040459,-0.194544,-0.167342,-0.124633,0.081722,0.820260,0.269346,-0.471731,1.319855,3.939966,2.490896
170,-0.275555,0.440890,-1.732979,-0.850345,-0.482157,0.958725,-3.387798,0.192882,-1.504021,0.203091,-1.323392
173,-0.494099,-0.088638,-0.167342,0.746221,0.739580,0.681794,-0.416411,3.050719,-0.966140,-0.917971,1.728038


In [18]:
feature_complete.loc[
    extreme_rows.index,
    ["participant_id", "session_id"] + FEATURES
]

,participant_id,session_id,eswan_adhd_inattention_total,eswan_adhd_hyperactivity_impulsivity_total,bissc_total,bas_total,im_average,em_average,cnb_cptnl_total_sen,cnb_cptnl_total_fp,cnb_edisc_tot,cnb_ddisc_tot,cnb_rdisc_tot
52,sub-20399,ses-1,-6.0,-15.0,15.0,32.0,1.571429,4.000000,0.3167,7.0,73.0,14.0,29.0
54,sub-20402,ses-1,4.0,0.0,20.0,39.0,3.428571,3.333333,0.7833,17.0,64.0,0.0,4.0
93,sub-20855,ses-1,1.0,6.0,18.0,29.0,5.428571,5.166667,0.3833,15.0,70.0,19.0,36.0
121,sub-20968,ses-1,6.0,12.0,20.0,21.0,5.714286,4.500000,0.9333,7.0,57.0,31.0,26.0
131,sub-21035,ses-1,-27.0,-27.0,10.0,13.0,2.285714,3.500000,0.7833,9.0,66.0,14.0,21.0
136,sub-21050,ses-1,-10.0,-1.0,23.0,51.0,1.428571,1.166667,0.6333,50.0,50.0,3.0,26.0
139,sub-21104,ses-1,5.0,19.0,18.0,23.0,3.142857,4.833333,0.1667,2.0,71.0,13.0,19.0
153,sub-21150,ses-1,-8.0,0.0,16.0,24.0,4.714286,5.833333,0.8833,11.0,73.0,32.0,41.0
170,sub-21516,ses-1,-1.0,6.0,10.0,19.0,3.857143,6.000000,0.3500,21.0,52.0,12.0,16.0
173,sub-21553,ses-1,-3.0,1.0,16.0,30.0,5.714286,5.666667,0.7833,64.0,56.0,6.0,36.0


In [ ]:
### Extreme-value inspection

Standardized values exceeding |3 SD| were inspected rather than
automatically excluded. Observations were retained unless there was
evidence of data error or implausibility, because extreme values may
represent meaningful phenotypic variation relevant to mixture modeling.

In [19]:
cluster_input = feature_complete[
    ["participant_id", "session_id"]
].copy()

for feature in FEATURES:
    cluster_input[feature] = X_z[feature].values

cluster_input.to_csv(
    OUTPUT_DIR / "clustering_input_z.csv",
    index=False
)

print(cluster_input.shape)
print("Saved clustering_input_z.csv")

(92, 13)
Saved clustering_input_z.csv
